# Feline Skin Disease Detection - Model Statistics & Graphs

Scores the cross-validation sweep produced by `colab_training_2.ipynb`.

Nothing is re-run here and **no model is ever loaded**: `train_one_run` already
wrote the test and validation probabilities for all 210 runs to Drive, and
`BaseClassifier.calibrate_and_evaluate` now reads those `.npy` files instead of
a `.keras` file. One call scores one (architecture, strategy, seed) across all 5
folds: it fits a temperature on each fold's *validation* probabilities, applies
it to that fold's *test* probabilities, then pools the 5 folds into one held-out
prediction per image.

That makes this a CPU-only notebook - it needs the repo, `fold_assignments.csv`,
and the `cv_run/` folder on Drive.

## What this notebook produces

| Output | Section | Feeds |
|---|---|---|
| accuracy, macro-F1 | 6 | overall results table |
| accuracy, macro-F1 per fold | 7 | fold-to-fold variation table |
| per-class precision, recall, F1 | 8 | per-class table (comment 5) |
| 5 temperature values per run | 9 | comment 7, evidence T was fit per run |
| ECE before and after | 6, 10 | comment 7 |
| bin counts before and after | 10 | comment 7, the sparse-bin complaint |
| Brier before and after | 6 | comment 7 |
| NLL before and after | 6 | comment 7 |
| confusion matrix | 11 | comment 12 figure |
| reliability diagrams | 12 | comment 7, visual companion to ECE |
| `y_true`, `y_prob`, `y_prob_cal` | 5, 13 | saved per run so anything else can be computed later without reloading |

Everything above is also written to CSV/NPZ under `cv_run/metrics/`, so the
tables can be pulled into the write-up without re-running the notebook.

## 1. Dependencies

`base_classifier.py` imports `ml_insights` at module scope for the reliability
diagrams, so it has to be installed even though most cells never plot one.

No GPU required - a CPU runtime is fine and starts faster.

In [ ]:
!pip install -q ml_insights

import tensorflow as tf

# TF is imported only for tf.math.confusion_matrix; nothing runs on the GPU here.
print("TensorFlow version:", tf.__version__)

## 2. Mount Google Drive & clone repo

In [ ]:
import os
import subprocess
import sys

from google.colab import drive
drive.mount('/content/drive')

# Ensure we're in a valid directory before cleanup
os.chdir('/content')
!rm -rf /content/repo

REPO_URL = "https://github.com/pelta-ai/feline-skin-disease-detection.git"
# Branch holding the probability-based calibrate_and_evaluate - must be pushed.
BRANCH = "update/calibrate-and-evaluate-from-preds"

# GIT_LFS_SKIP_SMUDGE keeps the clone from failing on LFS-tracked .keras files.
# We never open a model here, so pointer files are all we need.
clone = subprocess.run(
    ["git", "clone", "-b", BRANCH, REPO_URL, "/content/repo"],
    env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
    capture_output=True, text=True,
)
print(clone.stderr.strip())
if clone.returncode != 0:
    raise RuntimeError(f"Clone of branch '{BRANCH}' failed - has it been pushed to the remote?")
assert os.path.isdir('/content/repo/src'), "Checkout incomplete: /content/repo/src is missing."

os.chdir('/content/repo')
sys.path.insert(0, '/content/repo')
print("OK: repo at /content/repo")

## 3. Locate the sweep outputs

`colab_training_2.ipynb` wrote everything into one folder on Drive:

```
cv_run/preds_<arch>_<strategy>_f<k>_s<seed>.npy       test probabilities
cv_run/preds_val_<arch>_<strategy>_f<k>_s<seed>.npy   val probabilities (calibration)
cv_run/testset_f<k>.csv                               that fold's test rows, in prob order
cv_run/fold_assignments.csv                           the folds those runs were trained on
```

Use the `fold_assignments.csv` **copy inside `cv_run/`**, not the one in the
repo. `calibrate_and_evaluate` pairs val probabilities with val labels
positionally, so a regenerated fold file with different row order would silently
mis-label every validation image and fit garbage temperatures.

In [ ]:
import os
from collections import Counter

import pandas as pd

DRIVE_ROOT = "/content/drive/MyDrive/feline-skin-disease-detection"
PROBS_DIR = os.path.join(DRIVE_ROOT, "cv_run")

if not os.path.isdir(PROBS_DIR):
    raise FileNotFoundError(
        f"{PROBS_DIR} not found - run colab_training_2.ipynb first, or point "
        "PROBS_DIR at wherever that sweep wrote its probabilities."
    )

# The copy parked next to the probabilities is the authoritative one.
FOLD_CANDIDATES = [
    os.path.join(PROBS_DIR, "fold_assignments.csv"),
    os.path.join("src", "duplicate_image_audit", "fold_assignments.csv"),
    os.path.join(DRIVE_ROOT, "fold_assignments.csv"),
]
FOLD_CSV = next((p for p in FOLD_CANDIDATES if os.path.exists(p)), None)
if FOLD_CSV is None:
    raise FileNotFoundError(f"fold_assignments.csv not found. Looked in: {FOLD_CANDIDATES}")
if not FOLD_CSV.startswith(PROBS_DIR):
    print(f"WARNING: using {FOLD_CSV}, not the copy inside {PROBS_DIR}. "
          "If it was regenerated since training, the labels will not line up.")
print(f"Using folds from: {FOLD_CSV}")

folds = pd.read_csv(FOLD_CSV)
CLASS_NAMES = sorted(folds["label"].unique())
print(f"{len(folds)} rows, {len(CLASS_NAMES)} classes, {folds['fold'].nunique()} folds")
print(folds.groupby(["fold", "role"]).size().unstack(fill_value=0))

files = os.listdir(PROBS_DIR)
test_probs = [f for f in files if f.startswith("preds_") and not f.startswith("preds_val_")]
val_probs = [f for f in files if f.startswith("preds_val_")]
testsets = [f for f in files if f.startswith("testset_f")]
print(f"\n{len(test_probs)} test-prob files, {len(val_probs)} val-prob files, "
      f"{len(testsets)} testset CSVs")
per_arch = Counter(f[len("preds_"):].rsplit("_", 3)[0] for f in test_probs)
for a, n in sorted(per_arch.items()):
    print(f"  {a:22s} {n:3d}")

### Satisfy the dataset-path import

`src/data_manipulation/count_image_classes.py` calls `get_class_names(new_data/train)`
at *import* time, so importing any classifier fails if that folder is missing.
This notebook reads no images at all, so instead of mounting the 7.6k-image
dataset we create the empty class folders that import expects - the class order
this notebook actually uses comes from `set_class_names(folds)` below.

In [ ]:
from src.utils import constants

stub_train = os.path.join("/content/repo", constants.DATA_PATH, "train")
for name in CLASS_NAMES:
    os.makedirs(os.path.join(stub_train, name), exist_ok=True)
print(f"Stubbed {len(CLASS_NAMES)} class folders under {stub_train}")
print(CLASS_NAMES)

## 4. Configuration

`name=arch` matters: `self.name` is what `calibrate_and_evaluate` interpolates
into the `preds_*` filenames, exactly as `train_one_run` did when writing them.

`N_BINS=10` matches the new default in `expected_calibration_error`, which now
also returns the per-bin sample counts.

In [ ]:
import gc

import matplotlib.pyplot as plt
import ml_insights as mli
import numpy as np

from src.classifiers import ClassifierFactory
from src.classifiers.base_classifier import BaseClassifier

ARCHS = ["mobilenet_v2", "mobilenet_v3_small", "resnet50", "efficientnet_b0",
         "efficientnet_v2_b0", "nasnet_mobile", "convnext_tiny"]
STRATEGIES = ["frozen", "finetuned"]
SEEDS = (1, 2, 3)
N_FOLDS = folds["fold"].nunique()
N_BINS = 10

# Bin edges are (lo, hi], matching expected_calibration_error.
BIN_EDGES = np.linspace(0.0, 1.0, N_BINS + 1)
BIN_LABELS = [f"({lo:.1f},{hi:.1f}]" for lo, hi in zip(BIN_EDGES[:-1], BIN_EDGES[1:])]

OUT_DIR = os.path.join(PROBS_DIR, "metrics")
ARRAYS_DIR = os.path.join(OUT_DIR, "arrays")
os.makedirs(ARRAYS_DIR, exist_ok=True)

print(f"{len(ARCHS)} archs x {len(STRATEGIES)} strategies x {len(SEEDS)} seeds = "
      f"{len(ARCHS) * len(STRATEGIES) * len(SEEDS)} runs to score, "
      f"{N_FOLDS} folds pooled per run")
print(f"Writing tables to {OUT_DIR}")

## 5. Score every run

One `calibrate_and_evaluate` call per (arch, strategy, seed). It loops the 5
folds internally - fitting that fold's temperature on val, applying it to that
fold's test probabilities - and returns metrics over the pooled predictions,
which covers every image in the dataset exactly once.

Discrimination metrics (accuracy, macro-F1, confusion matrix, per-class P/R/F1)
are computed from the **uncalibrated** argmax. Temperature scaling is monotone,
so it cannot change them; only the calibration metrics move.

Each run's `y_true` / `y_prob` / `y_prob_cal` are saved to `metrics/arrays/`, so
any metric not tabulated below can be computed later without touching the
probabilities again.

In [ ]:
def run_is_complete(arch, strategy, seed):
    """Every fold of this run must have both prob files and its testset CSV."""
    for k in range(N_FOLDS):
        needed = (f"preds_{arch}_{strategy}_f{k}_s{seed}.npy",
                  f"preds_val_{arch}_{strategy}_f{k}_s{seed}.npy",
                  f"testset_f{k}.csv")
        if not all(os.path.exists(os.path.join(PROBS_DIR, f)) for f in needed):
            return False
    return True


results, skipped, failed = [], [], []

for arch in ARCHS:
    # Cheap: the constructor only records config, it does not build or download
    # a backbone (that happens in _build_model, which is never called here).
    clf = ClassifierFactory.create(arch, name=arch)
    clf.set_class_names(folds)

    for strategy in STRATEGIES:
        for seed in SEEDS:
            if not run_is_complete(arch, strategy, seed):
                skipped.append((arch, strategy, seed))
                continue

            try:
                r = clf.calibrate_and_evaluate(
                    probs_dir=PROBS_DIR,
                    strategy=strategy,
                    seed=seed,
                    fold_assignments_path=FOLD_CSV,
                    n_folds=N_FOLDS,
                    n_bins=N_BINS,
                    show_plots=False,
                )
            except Exception as e:
                # A length mismatch means the fold file no longer matches the
                # probabilities; record it rather than losing the whole sweep.
                failed.append((arch, strategy, seed, repr(e)))
                print(f"FAILED {arch} {strategy} s{seed}: {e}")
                continue

            np.savez_compressed(
                os.path.join(ARRAYS_DIR, f"{arch}_{strategy}_s{seed}.npz"),
                y_true=r["y_true"], y_prob=r["y_prob"], y_prob_cal=r["y_prob_cal"],
                confusion_matrix=r["confusion_matrix"],
                temperatures=np.array(r["temperatures"]),
            )
            results.append(r)

            print(f"{arch:20s} {strategy:10s} s{seed}  n={r['n_images']:5d}  "
                  f"acc={r['accuracy']:.4f}  macroF1={r['macro_f1']:.4f}  "
                  f"ECE {r['ece_before']:.4f}->{r['ece_after']:.4f}  "
                  f"NLL {r['nll_before']:.4f}->{r['nll_after']:.4f}  "
                  f"T={np.mean(r['temperatures']):.3f}")

    del clf
    gc.collect()

print(f"\nScored {len(results)} runs, skipped {len(skipped)} (files absent), "
      f"{len(failed)} failed")
if skipped:
    print("skipped:", skipped)

## 6. Overall results table

Accuracy and macro-F1, plus every calibration metric before and after temperature
scaling. `runs_df` is one row per (arch, strategy, seed) with the 5 fold
temperatures spread across `T_f0..T_f4`; `overall_df` averages across seeds.

Negative `ece_delta` / `nll_delta` / `brier_delta` mean calibration improved.
Note that NLL and Brier can move in opposite directions to ECE: temperature
scaling optimises NLL on validation data, so a worse Brier alongside a better
NLL is a real result, not a bug.

In [ ]:
rows = []
for r in results:
    row = {
        "arch": r["arch"], "strategy": r["strategy"], "seed": r["seed"],
        "n_images": r["n_images"],
        "accuracy": r["accuracy"], "macro_f1": r["macro_f1"],
        "ece_before": r["ece_before"], "ece_after": r["ece_after"],
        "brier_before": r["brier_before"], "brier_after": r["brier_after"],
        "nll_before": r["nll_before"], "nll_after": r["nll_after"],
        "T_mean": float(np.mean(r["temperatures"])),
        "T_min": float(np.min(r["temperatures"])),
        "T_max": float(np.max(r["temperatures"])),
    }
    for k, T in enumerate(r["temperatures"]):
        row[f"T_f{k}"] = T
    rows.append(row)

runs_df = pd.DataFrame(rows)
for metric in ("ece", "brier", "nll"):
    runs_df[f"{metric}_delta"] = runs_df[f"{metric}_after"] - runs_df[f"{metric}_before"]

runs_df = runs_df.sort_values(["arch", "strategy", "seed"]).reset_index(drop=True)
runs_df.to_csv(os.path.join(OUT_DIR, "per_run_metrics.csv"), index=False)

pd.set_option("display.width", 220, "display.max_columns", 60)
print("Per-run metrics (one row per arch x strategy x seed):")
print(runs_df.round(4).to_string(index=False))

In [ ]:
METRICS = ["accuracy", "macro_f1", "ece_before", "ece_after", "brier_before",
           "brier_after", "nll_before", "nll_after", "T_mean"]

overall_df = (runs_df
              .groupby(["arch", "strategy"])
              .agg(seeds=("seed", "count"),
                   **{f"{m}_{stat}": (m, stat)
                      for m in METRICS for stat in ("mean", "std")})
              .reset_index())
overall_df.to_csv(os.path.join(OUT_DIR, "overall_metrics.csv"), index=False)

# Compact "mean +/- std" view for the write-up.
def pm(df, m, dp=4):
    return df[f"{m}_mean"].map(lambda v: f"{v:.{dp}f}") + " +/- " + \
           df[f"{m}_std"].fillna(0).map(lambda v: f"{v:.{dp}f}")

report = pd.DataFrame({
    "arch": overall_df["arch"], "strategy": overall_df["strategy"],
    "seeds": overall_df["seeds"],
    "accuracy": pm(overall_df, "accuracy"), "macro_F1": pm(overall_df, "macro_f1"),
    "ECE_before": pm(overall_df, "ece_before"), "ECE_after": pm(overall_df, "ece_after"),
    "Brier_before": pm(overall_df, "brier_before"), "Brier_after": pm(overall_df, "brier_after"),
    "NLL_before": pm(overall_df, "nll_before"), "NLL_after": pm(overall_df, "nll_after"),
    "T": pm(overall_df, "T_mean", 3),
})
report.to_csv(os.path.join(OUT_DIR, "overall_metrics_report.csv"), index=False)
print("Mean +/- std across seeds:")
print(report.to_string(index=False))

## 7. Per-fold accuracy and macro-F1

Section 6 pools the 5 folds into one held-out prediction per image, which hides
how much the score moves *between* folds. This splits the pooled arrays back
apart and scores each fold on its own - accuracy and macro-F1 for fold 0, fold 1,
and so on - so fold-to-fold variation can be quoted alongside seed-to-seed
variation.

The split is positional: `calibrate_and_evaluate` concatenates the folds in
order `f0..f{N-1}`, each block exactly as long as that fold's `testset_f<k>.csv`,
so those same CSVs give the boundaries back. Every run is length-checked against
that total before it is sliced.

As in section 6 these are computed from the **uncalibrated** argmax - temperature
scaling is monotone and cannot move them.

Two views come out of it:

- **by fold, over every run** - is one fold simply harder than the others?
- **by arch x strategy x fold** - does a given model swing across folds, and how
  does the mean of its 5 folds compare with its pooled score?

Those two numbers are not identical by construction: the pooled score weights
every image equally, while the fold mean weights every fold equally, and macro-F1
is not linear in the counts anyway.

In [ ]:
FOLD_SIZES = [len(pd.read_csv(os.path.join(PROBS_DIR, f"testset_f{k}.csv")))
              for k in range(N_FOLDS)]
FOLD_BOUNDS = np.cumsum([0] + FOLD_SIZES)
print("Test images per fold: "
      + ", ".join(f"f{k}={n}" for k, n in enumerate(FOLD_SIZES))
      + f"  (total {FOLD_BOUNDS[-1]})")

# One throwaway classifier so a fold is scored by exactly the same helpers as the
# pooled run - same confusion matrix, same eps, same macro-F1 definition.
scorer = ClassifierFactory.create(ARCHS[0], name=ARCHS[0])
scorer.set_class_names(folds)

fold_rows = []
for r in results:
    if len(r["y_true"]) != FOLD_BOUNDS[-1]:
        raise ValueError(
            f"{r['arch']} {r['strategy']} s{r['seed']}: pooled length "
            f"{len(r['y_true'])} != sum of fold sizes {FOLD_BOUNDS[-1]}; the "
            "testset CSVs no longer describe how these probabilities were pooled."
        )

    y_true = BaseClassifier._to_int(r["y_true"])
    y_pred = r["y_prob"].argmax(1)
    for k in range(N_FOLDS):
        lo, hi = FOLD_BOUNDS[k], FOLD_BOUNDS[k + 1]
        cm = scorer._confusion_matrix(y_true[lo:hi], y_pred[lo:hi])
        acc, _, _, _, macro_f1 = scorer._metrics_from_confusion_matrix(cm)
        fold_rows.append({
            "arch": r["arch"], "strategy": r["strategy"], "seed": r["seed"],
            "fold": k, "n_images": int(hi - lo),
            "accuracy": acc, "macro_f1": macro_f1,
        })

per_fold_df = (pd.DataFrame(fold_rows)
               .sort_values(["arch", "strategy", "seed", "fold"])
               .reset_index(drop=True))
per_fold_df.to_csv(os.path.join(OUT_DIR, "per_fold_metrics.csv"), index=False)
print(f"\nScored {len(per_fold_df)} fold-level results "
      f"({len(results)} runs x {N_FOLDS} folds)")
print(per_fold_df.round(4).head(2 * N_FOLDS).to_string(index=False))

In [ ]:
# View 1: how each fold scores across every run, i.e. is one fold harder?
by_fold = (per_fold_df
           .groupby("fold")
           .agg(runs=("accuracy", "count"), n_images=("n_images", "first"),
                accuracy_mean=("accuracy", "mean"), accuracy_std=("accuracy", "std"),
                macro_f1_mean=("macro_f1", "mean"), macro_f1_std=("macro_f1", "std"))
           .reset_index())
by_fold.to_csv(os.path.join(OUT_DIR, "per_fold_metrics_by_fold.csv"), index=False)

print(f"Mean +/- std over all {len(results)} runs, one row per fold:")
print(pd.DataFrame({
    "fold": by_fold["fold"], "runs": by_fold["runs"], "n_images": by_fold["n_images"],
    "accuracy": pm(by_fold, "accuracy"), "macro_F1": pm(by_fold, "macro_f1"),
}).to_string(index=False))

# View 2: same split, but per arch x strategy and averaged over seeds.
by_combo_fold = (per_fold_df
                 .groupby(["arch", "strategy", "fold"])
                 .agg(accuracy=("accuracy", "mean"), macro_f1=("macro_f1", "mean"))
                 .reset_index())
by_combo_fold.to_csv(os.path.join(OUT_DIR, "per_fold_metrics_by_arch.csv"), index=False)

pooled_scores = overall_df.set_index(["arch", "strategy"])
for metric, label in (("accuracy", "Accuracy"), ("macro_f1", "Macro-F1")):
    wide_fold = (by_combo_fold
                 .pivot(index=["arch", "strategy"], columns="fold", values=metric)
                 .rename(columns=lambda k: f"fold{k}")
                 .rename_axis(columns=None))
    fold_cols = wide_fold.filter(like="fold")
    wide_fold["fold_mean"] = fold_cols.mean(axis=1)
    wide_fold["fold_std"] = fold_cols.std(axis=1)
    wide_fold["fold_spread"] = fold_cols.max(axis=1) - fold_cols.min(axis=1)
    # pooled = section 6's number, which weights images equally rather than folds.
    wide_fold["pooled"] = pooled_scores[f"{metric}_mean"]
    wide_fold.to_csv(os.path.join(OUT_DIR, f"per_fold_{metric}_by_arch.csv"))

    print(f"\n{label} per fold (mean over seeds), with the pooled score for comparison:")
    print(wide_fold.round(4).to_string())

## 8. Per-class table (comment 5)

Precision, recall and F1 for every class, averaged over seeds. `support` is the
number of held-out images of that class in one pooled run - i.e. the whole
dataset, since the 5 test folds partition it.

In [ ]:
per_class_rows = []
for r in results:
    cm = np.asarray(r["confusion_matrix"])
    for i, name in enumerate(CLASS_NAMES):
        per_class_rows.append({
            "arch": r["arch"], "strategy": r["strategy"], "seed": r["seed"],
            "class": name, "support": int(cm[i].sum()),
            "precision": float(r["per_class_precision"][i]),
            "recall": float(r["per_class_recall"][i]),
            "f1": float(r["per_class_f1"][i]),
        })

per_class_df = pd.DataFrame(per_class_rows)
per_class_df.to_csv(os.path.join(OUT_DIR, "per_class_metrics_per_run.csv"), index=False)

per_class_mean = (per_class_df
                  .groupby(["arch", "strategy", "class"])
                  .agg(support=("support", "mean"),
                       precision_mean=("precision", "mean"), precision_std=("precision", "std"),
                       recall_mean=("recall", "mean"), recall_std=("recall", "std"),
                       f1_mean=("f1", "mean"), f1_std=("f1", "std"))
                  .reset_index())
per_class_mean.to_csv(os.path.join(OUT_DIR, "per_class_metrics.csv"), index=False)

for (arch, strategy), block in per_class_mean.groupby(["arch", "strategy"], sort=False):
    print(f"\n--- {arch} ({strategy}) ---")
    table = pd.DataFrame({
        "class": block["class"],
        "support": block["support"].astype(int),
        "precision": pm(block, "precision", 3),
        "recall": pm(block, "recall", 3),
        "F1": pm(block, "f1", 3),
    })
    print(table.to_string(index=False))

## 9. Temperatures (comment 7)

Five values per run, one per fold, each fitted on that fold's validation
probabilities alone and applied only to that fold's test probabilities. The
spread across folds is the evidence that T was refit per run rather than reused
from a single global fit.

`T > 1` means the raw softmax was overconfident and got softened; `T < 1` means
it was underconfident and got sharpened.

In [ ]:
temp_rows = [
    {"arch": r["arch"], "strategy": r["strategy"], "seed": r["seed"],
     "fold": k, "temperature": float(T)}
    for r in results for k, T in enumerate(r["temperatures"])
]
temps_df = pd.DataFrame(temp_rows)
temps_df.to_csv(os.path.join(OUT_DIR, "temperatures.csv"), index=False)

wide = (temps_df
        .pivot_table(index=["arch", "strategy", "seed"], columns="fold", values="temperature")
        .rename(columns=lambda k: f"fold{k}"))
wide["mean"] = wide.mean(axis=1)
wide["spread"] = wide.filter(like="fold").max(axis=1) - wide.filter(like="fold").min(axis=1)
print("Fitted temperature per fold (5 per run):")
print(wide.round(3).to_string())

print("\nBy arch x strategy, over all folds and seeds:")
print(temps_df.groupby(["arch", "strategy"])["temperature"]
      .agg(n="count", mean="mean", std="std", min="min", max="max")
      .round(3).to_string())

## 10. Calibration bins (comment 7)

ECE is a weighted average over 10 confidence bins, so it is only as trustworthy
as the bins are populated. These are the per-bin sample counts before and after
scaling - the direct answer to the sparse-bin complaint.

`n_bins_nonempty` and `pct_in_top_bin` are the two numbers worth quoting: if
almost everything sits in `(0.9,1.0]`, the ECE is effectively being estimated
from one bin, and temperature scaling's main visible effect is redistributing
mass into the middle bins where the estimate is better conditioned.

In [ ]:
bin_rows = []
for r in results:
    n = r["n_images"]
    for b, (before, after) in enumerate(zip(r["bin_counts_before"], r["bin_counts_after"])):
        bin_rows.append({
            "arch": r["arch"], "strategy": r["strategy"], "seed": r["seed"],
            "bin": b, "bin_range": BIN_LABELS[b],
            "count_before": int(before), "count_after": int(after),
            "pct_before": 100.0 * before / n, "pct_after": 100.0 * after / n,
        })

bins_df = pd.DataFrame(bin_rows)
bins_df.to_csv(os.path.join(OUT_DIR, "calibration_bin_counts.csv"), index=False)

# Pooled over seeds: total images per bin for each arch x strategy.
pooled = (bins_df.groupby(["arch", "strategy", "bin", "bin_range"], as_index=False)
          [["count_before", "count_after"]].sum())
before_wide = pooled.pivot(index=["arch", "strategy"], columns="bin_range",
                           values="count_before")[BIN_LABELS]
after_wide = pooled.pivot(index=["arch", "strategy"], columns="bin_range",
                          values="count_after")[BIN_LABELS]
print("Bin counts BEFORE calibration (summed over seeds):")
print(before_wide.astype(int).to_string())
print("\nBin counts AFTER calibration (summed over seeds):")
print(after_wide.astype(int).to_string())

# How many bins actually carry any mass, and how much of it piles into the last one.
nonempty = (pooled.groupby(["arch", "strategy"])
            .agg(n_bins_nonempty_before=("count_before", lambda s: int((s > 0).sum())),
                 n_bins_nonempty_after=("count_after", lambda s: int((s > 0).sum())))
            .reset_index())
top_bin = (bins_df[bins_df["bin"] == N_BINS - 1]
           .groupby(["arch", "strategy"])
           .agg(pct_in_top_bin_before=("pct_before", "mean"),
                pct_in_top_bin_after=("pct_after", "mean"))
           .reset_index())
occupancy = nonempty.merge(top_bin, on=["arch", "strategy"])
occupancy.to_csv(os.path.join(OUT_DIR, "calibration_bin_occupancy.csv"), index=False)
print("\nBin occupancy (how much of the ECE rests on how few bins):")
print(occupancy.round(2).to_string(index=False))

In [ ]:
# Same numbers as a figure: bin mass before vs after, per arch x strategy.
combos = list(before_wide.index)
ncols = 2
nrows = int(np.ceil(len(combos) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(14, 3.2 * nrows), squeeze=False)
x = np.arange(N_BINS)

for ax, (arch, strategy) in zip(axes.ravel(), combos):
    ax.bar(x - 0.2, before_wide.loc[(arch, strategy)].values, width=0.4, label="before")
    ax.bar(x + 0.2, after_wide.loc[(arch, strategy)].values, width=0.4, label="after")
    ax.set_title(f"{arch} ({strategy})", fontsize=10)
    ax.set_xticks(x)
    ax.set_xticklabels(BIN_LABELS, rotation=60, fontsize=7)
    ax.set_ylabel("images")
    ax.legend(fontsize=8)

for ax in axes.ravel()[len(combos):]:
    ax.axis("off")

fig.suptitle("Confidence-bin occupancy before vs after temperature scaling")
fig.tight_layout()
plt.show()

## 11. Confusion matrices (comment 12)

Mean confusion matrix over seeds for each arch x strategy, over the pooled
held-out predictions. Counts are seed-averaged, so entries are fractional.

Set `SHOW_NORMALIZED_CM = True` for the row-normalised version (each row sums to
100%, so the diagonal reads as per-class recall).

In [ ]:
SHOW_NORMALIZED_CM = False

# display_confusion_matrix is an instance method that only needs class_names,
# so one throwaway classifier can render every matrix.
viz = ClassifierFactory.create(ARCHS[0], name=ARCHS[0])
viz.set_class_names(folds)

by_combo = {}
for r in results:
    by_combo.setdefault((r["arch"], r["strategy"]), []).append(r["confusion_matrix"])

for (arch, strategy), cms in by_combo.items():
    mean_cm = np.mean(cms, axis=0)
    pd.DataFrame(mean_cm, index=CLASS_NAMES, columns=CLASS_NAMES).to_csv(
        os.path.join(OUT_DIR, f"confusion_matrix_{arch}_{strategy}.csv"))

    viz.display_confusion_matrix(
        mean_cm,
        title=f"{arch} - {strategy}\nmean of {len(cms)} seeds, pooled over {N_FOLDS} folds",
    )

    if SHOW_NORMALIZED_CM:
        norm_cm = 100.0 * mean_cm / mean_cm.sum(axis=1, keepdims=True)
        viz.display_confusion_matrix(
            norm_cm,
            title=f"{arch} - {strategy} (row-normalised %)\nmean of {len(cms)} seeds",
        )

## 12. Reliability diagrams

The visual companion to the ECE numbers, pooled across seeds so each curve is
backed by every held-out prediction the architecture produced. The histogram
under each curve is the same bin occupancy tabulated in section 10.

In [ ]:
PLOT_RELIABILITY = True

if PLOT_RELIABILITY:
    pooled_runs = {}
    for r in results:
        pooled_runs.setdefault((r["arch"], r["strategy"]), []).append(r)

    for (arch, strategy), runs in pooled_runs.items():
        y_true = np.concatenate([r["y_true"] for r in runs])
        y_prob = np.concatenate([r["y_prob"] for r in runs])
        y_prob_cal = np.concatenate([r["y_prob_cal"] for r in runs])
        correct = (y_prob.argmax(1) == BaseClassifier._to_int(y_true)).astype(int)

        BaseClassifier.display_reliability_diagram(
            mli.plot_reliability_diagram(correct, y_prob.max(1), show_histogram=True),
            f"{arch} - {strategy}: before calibration (pool of {len(runs)} seeds)")
        BaseClassifier.display_reliability_diagram(
            mli.plot_reliability_diagram(correct, y_prob_cal.max(1), show_histogram=True),
            f"{arch} - {strategy}: after calibration (pool of {len(runs)} seeds)")

## 13. Saved artifacts

Everything above lives under `cv_run/metrics/`. The `arrays/` folder holds the
raw `y_true` / `y_prob` / `y_prob_cal` per run, so any metric that is not
tabulated here can be computed later without re-reading the sweep:

```python
d = np.load(".../cv_run/metrics/arrays/mobilenet_v2_frozen_s1.npz")
d["y_true"], d["y_prob"], d["y_prob_cal"], d["temperatures"], d["confusion_matrix"]
```

In [ ]:
print(f"{OUT_DIR}\n")
for name in sorted(os.listdir(OUT_DIR)):
    path = os.path.join(OUT_DIR, name)
    if os.path.isdir(path):
        n = len(os.listdir(path))
        size = sum(os.path.getsize(os.path.join(path, f)) for f in os.listdir(path))
        print(f"  {name + '/':45s} {n:3d} files  {size / 1e6:7.2f} MB")
    else:
        print(f"  {name:45s} {os.path.getsize(path) / 1e3:7.1f} KB")

# Round-trip one run to prove the arrays reload cleanly.
sample = sorted(os.listdir(ARRAYS_DIR))[0]
d = np.load(os.path.join(ARRAYS_DIR, sample))
print(f"\n{sample}: " + ", ".join(f"{k}{d[k].shape}" for k in d.files))